In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.append("..")

from src.data.cleaning import (
    clean_vehicle_data,
    create_shared_split
)

DATA_PATH = Path("../data/raw/car_price_dataset.csv")

In [2]:
df = pd.read_csv(DATA_PATH)
df_clean = clean_vehicle_data(df)

X_train, X_test, y_train, y_test = create_shared_split(df_clean)

print("Cleaned shape:", df_clean.shape)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Cleaned shape: (9743, 16)
X_train: (7794, 15)
X_test: (1949, 15)
y_train: (7794,)
y_test: (1949,)


In [3]:
# Inspect columns before feature engineering
print(X_train.columns.tolist())
display(X_train.head())

# Check YOM-Mileage redundancy
print(X_train[["YOM", "Mileage(KM)"]].corr())

mileage_per_year = X_train.groupby("YOM")["Mileage(KM)"].nunique()

print("Maximum unique mileage values per YOM:", mileage_per_year.max())
print("Years with more than one mileage value:", (mileage_per_year > 1).sum())

['Brand', 'Model', 'YOM', 'Engine (cc)', 'Gear', 'Fuel Type', 'Mileage(KM)', 'Town', 'Date', 'Leasing', 'Condition', 'AIR CONDITION', 'POWER STEERING', 'POWER MIRROR', 'POWER WINDOW']


,Brand,Model,YOM,Engine (cc),Gear,Fuel Type,Mileage(KM),Town,Date,Leasing,Condition,AIR CONDITION,POWER STEERING,POWER MIRROR,POWER WINDOW
8594,TOYOTA,PREMIO,2019.0,1500.0,Automatic,Petrol,66000.0,Kadawatha,2024-12-30,No Leasing,USED,Available,Available,Available,Available
6080,SUZUKI,WAGON R FZ,2014.0,660.0,Automatic,Petrol,121000.0,Colombo,2024-12-21,No Leasing,USED,Available,Available,Available,Available
4648,SUZUKI,ALTO JAPAN,2003.0,660.0,Automatic,Petrol,242000.0,Kesbewa,2025-01-28,No Leasing,USED,Available,Available,Available,Available
6458,SUZUKI,WAGON R STINGRAY J STYLE,2015.0,800.0,Automatic,Hybrid,110000.0,Negombo,2025-01-11,No Leasing,USED,Available,Available,Available,Available
6441,SUZUKI,WAGON R STINGRAY,2015.0,660.0,Automatic,Petrol,110000.0,Matara,2024-12-04,No Leasing,USED,Available,Available,Available,Available


             YOM  Mileage(KM)
YOM          1.0         -1.0
Mileage(KM) -1.0          1.0
Maximum unique mileage values per YOM: 1
Years with more than one mileage value: 0


In [4]:
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

# Vehicle age
X_train_fe["Vehicle_Age"] = X_train_fe["Date"].dt.year - X_train_fe["YOM"]
X_test_fe["Vehicle_Age"] = X_test_fe["Date"].dt.year - X_test_fe["YOM"]

# Listing month
X_train_fe["Listing_Month"] = X_train_fe["Date"].dt.month
X_test_fe["Listing_Month"] = X_test_fe["Date"].dt.month

# Number of available vehicle features
feature_cols = [
    "AIR CONDITION",
    "POWER STEERING",
    "POWER MIRROR",
    "POWER WINDOW"
]

X_train_fe["Feature_Count"] = (
    X_train_fe[feature_cols] == "Available"
).sum(axis=1)

X_test_fe["Feature_Count"] = (
    X_test_fe[feature_cols] == "Available"
).sum(axis=1)

# Remove redundant/raw columns
X_train_fe = X_train_fe.drop(
    columns=["YOM", "Mileage(KM)", "Date"]
)

X_test_fe = X_test_fe.drop(
    columns=["YOM", "Mileage(KM)", "Date"]
)

In [5]:
print("Train shape:", X_train_fe.shape)
print("Test shape:", X_test_fe.shape)

print(X_train_fe.columns.tolist())

display(
    X_train_fe[
        ["Brand", "Model", "Vehicle_Age",
         "Listing_Month", "Feature_Count"]
    ].head()
)

Train shape: (7794, 15)
Test shape: (1949, 15)
['Brand', 'Model', 'Engine (cc)', 'Gear', 'Fuel Type', 'Town', 'Leasing', 'Condition', 'AIR CONDITION', 'POWER STEERING', 'POWER MIRROR', 'POWER WINDOW', 'Vehicle_Age', 'Listing_Month', 'Feature_Count']


,Brand,Model,Vehicle_Age,Listing_Month,Feature_Count
8594,TOYOTA,PREMIO,5.0,12,4
6080,SUZUKI,WAGON R FZ,10.0,12,4
4648,SUZUKI,ALTO JAPAN,22.0,1,4
6458,SUZUKI,WAGON R STINGRAY J STYLE,10.0,1,4
6441,SUZUKI,WAGON R STINGRAY,9.0,12,4


## Reusable Feature Engineering Function

The finalized feature-engineering logic was moved to `src/features/engineering.py` so the same transformations can be reused consistently in later preprocessing and modelling stages.

In [6]:
from src.features.engineering import engineer_vehicle_features

X_train_engineered = engineer_vehicle_features(X_train)
X_test_engineered = engineer_vehicle_features(X_test)

print("Train shape:", X_train_engineered.shape)
print("Test shape:", X_test_engineered.shape)

print(X_train_engineered.columns.tolist())

display(X_train_engineered.head())

Train shape: (7794, 15)
Test shape: (1949, 15)
['Brand', 'Model', 'Engine (cc)', 'Gear', 'Fuel Type', 'Town', 'Leasing', 'Condition', 'AIR CONDITION', 'POWER STEERING', 'POWER MIRROR', 'POWER WINDOW', 'Vehicle_Age', 'Listing_Month', 'Feature_Count']


,Brand,Model,Engine (cc),Gear,Fuel Type,Town,Leasing,Condition,AIR CONDITION,POWER STEERING,POWER MIRROR,POWER WINDOW,Vehicle_Age,Listing_Month,Feature_Count
8594,TOYOTA,PREMIO,1500.0,Automatic,Petrol,Kadawatha,No Leasing,USED,Available,Available,Available,Available,5.0,12,4
6080,SUZUKI,WAGON R FZ,660.0,Automatic,Petrol,Colombo,No Leasing,USED,Available,Available,Available,Available,10.0,12,4
4648,SUZUKI,ALTO JAPAN,660.0,Automatic,Petrol,Kesbewa,No Leasing,USED,Available,Available,Available,Available,22.0,1,4
6458,SUZUKI,WAGON R STINGRAY J STYLE,800.0,Automatic,Hybrid,Negombo,No Leasing,USED,Available,Available,Available,Available,10.0,1,4
6441,SUZUKI,WAGON R STINGRAY,660.0,Automatic,Petrol,Matara,No Leasing,USED,Available,Available,Available,Available,9.0,12,4


In [7]:
print("Feature_Count:")
print(X_train_engineered["Feature_Count"].value_counts().sort_index())

print("\nListing_Month:")
print(X_train_engineered["Listing_Month"].value_counts().sort_index())

print("\nVehicle_Age:")
print(X_train_engineered["Vehicle_Age"].describe())

Feature_Count:
Feature_Count
1     666
2     659
3     967
4    5502
Name: count, dtype: int64

Listing_Month:
Listing_Month
1     3579
2     1925
12    2290
Name: count, dtype: int64

Vehicle_Age:
count    7787.000000
mean       17.972390
std         9.874014
min         0.000000
25%        10.000000
50%        15.000000
75%        24.000000
max        69.000000
Name: Vehicle_Age, dtype: float64


## Feature Engineering Decisions

Three new features were created:

- **Vehicle_Age** – calculated from the listing year and YOM to represent the vehicle's age at the time of listing.
- **Listing_Month** – extracted from Date to retain simple listing-time information.
- **Feature_Count** – counts how many of the four listed vehicle convenience features are available.

The following raw features were removed:

- **Mileage(KM)** – removed because it had a perfect negative correlation (-1.0) with YOM and each YOM corresponded to only one mileage value in the training data, making it redundant.
- **YOM** – removed after deriving Vehicle_Age, which provides a more interpretable representation for vehicle pricing.
- **Date** – removed after extracting useful date-based information.

Missing values were not handled at this stage because imputation is part of the next preprocessing stage.